# 01_load_data.ipynb

**Project:** Hydraulic Condition Monitoring Dashboard  
**Stage:** Data Ingestion (Step 01)

---

### 🎯 Objective
This notebook prepares the **canonical dataset** for the entire project.  
It:
- Reads all raw `.txt` sensor files and the system profile.
- Builds a unified feature matrix `X` and aligned label vector `y`.
- Runs integrity checks for consistency.
- Exports standardized artifacts for later notebooks.

---

### 📦 Outputs (official contract for later stages)
| File | Description |
|------|--------------|
| `data/processed/X_features.parquet` | Clean feature matrix (merged sensors) |
| `data/processed/y_labels.parquet` | Aligned condition labels |
| `data/metadata/features_index.csv` | Feature dictionary with sensor grouping info |

> From this point forward, **no other notebook should import `data_io.py` or reload raw `.txt`**.  
> All subsequent steps (`02_eda`, `03_feature_engineering`, `04_modeling`) start from these artifacts.


In [1]:
# === Imports & path setup ===
from pathlib import Path
import sys
import pandas as pd

# --- locate and import utils/data_io.py ---
repo_root = Path.cwd()
if not (repo_root / "utils" / "data_io.py").exists():
    repo_root = Path.cwd().parents[0]
sys.path.insert(0, str(repo_root))

from utils.data_io import (
    ensure_dirs,
    load_profile,
    load_all_sensors,
    build_features_matrix,
    save_processed,
    write_features_index,
    ROOT_DIR,
    RAW_DIR,
    META_DIR,
    PROC_DIR,
)

## 1. Project directories

We standardize the structure:

data/
├── raw/ → all original .txt sensor files
├── processed/ → parquet datasets for EDA/modeling
└── metadata/ → small reference CSVs (feature index, mappings)


If these directories don’t exist, they’ll be created automatically.                        

In [2]:
print("ROOT_DIR :", ROOT_DIR)
print("RAW_DIR  :", RAW_DIR)
print("META_DIR :", META_DIR)
print("PROC_DIR :", PROC_DIR)

ensure_dirs()
print("✅ Directories verified/created.")


ROOT_DIR : C:\Users\melny\hydraulic_dashboard
RAW_DIR  : C:\Users\melny\hydraulic_dashboard\data\raw
META_DIR : C:\Users\melny\hydraulic_dashboard\data\metadata
PROC_DIR : C:\Users\melny\hydraulic_dashboard\data\processed
✅ Directories verified/created.


## 2. Load profile / labels

`load_profile()` reads the system profile (often `profile.txt`)  
and returns a DataFrame with operational or failure states for each observation.

This will serve as our `y` (target labels) later.

In [3]:
labels_df = load_profile()
print("labels_df shape:", labels_df.shape)
display(labels_df.head())

✅ Loaded profile.txt | shape=(2205, 5) | from=data\metadata\profile.txt
labels_df shape: (2205, 5)


,0,1,2,3,4
0,3,100,0,130,1
1,3,100,0,130,1
2,3,100,0,130,1
3,3,100,0,130,1
4,3,100,0,130,1


## Raw Sensor Schema Example

To ensure the parsing process is correct, the table below shows the **schema of one raw `.txt` sensor file`** before any cleaning or transformation.

Typical raw files contain:

- Timestamp or cycle index  
- Pressure sensor channels (PS1–PS6)  
- Vibration sensor channels (VS1–VS4)  
- Temperature channels (TS1–TS2)  
- Flow or efficiency metrics (FS1, EPS1, etc.)  
- The condition labels for each subsystem (Cooler, Valve, Pump, Accumulator, Stability)

This preview helps validate:
- Column names match expected metadata  
- Number of columns is consistent across all raw files  
- There are no unexpected extra/missing columns


## 3. Load raw sensor data

`load_all_sensors()`:
- Scans `data/raw/` for `.txt` files.
- Handles mixed delimiters and encodings.
- Returns:
  - `sensor_frames`: dict `{sensor_name: DataFrame}`
  - `sensor_order`: deterministic order for consistent column arrangement


In [ ]:
sensor_frames, sensor_order = load_all_sensors()

print(f"Loaded {len(sensor_frames)} sensor frames.")
print("First few sensors:", list(sensor_frames.keys())[:5])
print("Order preview:", sensor_order[:5])

first_key = sensor_order[0] if sensor_order else None
if first_key:
    print(f"Example sensor '{first_key}' shape:", sensor_frames[first_key].shape)
    display(sensor_frames[first_key].head())


— Loading sensor files —
   Loaded CE.txt         | shape=(2205, 60)     | delim=whitespace
   Loaded CP.txt         | shape=(2205, 60)     | delim=whitespace
   Loaded EPS1.txt       | shape=(2205, 6000)   | delim=whitespace
   Loaded FS1.txt        | shape=(2205, 600)    | delim=whitespace
   Loaded FS2.txt        | shape=(2205, 600)    | delim=whitespace


In [ ]:
print(f"📄 Total raw sensor files loaded: {len(sensor_frames)}")

## 4. Build features matrix `X` and target vector `y`

`build_features_matrix()`:
- Concatenates all sensor frames horizontally into one DataFrame `X`.
- Aligns and merges `labels_df` into `y`.
- Ensures rows correspond across all tables.

Result:
- `X`: features for modeling  
- `y`: target condition/health state


In [ ]:
X, y = build_features_matrix(sensor_frames, sensor_order, labels_df)

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())
display(y.head())


## 5. Integrity checks

Before saving, we must confirm perfect alignment.

Checks:
1. Each sensor file has the same number of rows.
2. `X` and `y` have equal lengths.
3. All shapes match the expected total (e.g. 2205).

If anything fails, we fix the source issue now — not later.


In [ ]:
EXPECTED_ROWS = 2205  # adjust if dataset differs

# Identify mismatched sensors
row_mismatch = [
    (name, df.shape) for name, df in sensor_frames.items()
    if df.shape[0] != EXPECTED_ROWS
]
print("Row mismatches:", "none" if not row_mismatch else row_mismatch)

# Core assertions
assert len(X) == len(y), f"Mismatch: X({len(X)}) vs y({len(y)})"

if EXPECTED_ROWS:
    assert len(X) == EXPECTED_ROWS, f"Unexpected X rows: {len(X)}"
    assert len(y) == EXPECTED_ROWS, f"Unexpected y rows: {len(y)}"

for s_name, s_df in sensor_frames.items():
    assert s_df.shape[0] == len(X), f"{s_name}: {s_df.shape[0]} != {len(X)}"

print("✅ Integrity check passed.")


## 6. Save canonical artifacts

We now freeze this clean dataset.

Artifacts to be created:

| File | Purpose |
|------|----------|
| `data/processed/X_features.parquet` | canonical feature matrix |
| `data/processed/y_labels.parquet` | aligned labels |
| `data/metadata/features_index.csv` | dictionary of features and sensor groups |

Later notebooks will only use these.


In [ ]:
# Save feature matrix and labels
x_path, y_path = save_processed(X, y)
print("Saved processed features to:", x_path)
print("Saved labels to:", y_path)

# Save feature index metadata
meta_idx_path = write_features_index(X)
print("Saved feature index to:", meta_idx_path)

## 7. Pipeline contract for downstream notebooks

**Artifacts produced by this notebook:**
- `data/processed/X_processed.parquet`
- `data/processed/y_labels.parquet`
- `data/metadata/features_index.csv`

---

### 🔒 Usage rules
1. **Do NOT** import or run `utils/data_io.py` again.
2. **02_eda.ipynb** → loads parquet + feature index.
3. **03_feature_engineering.ipynb** → starts from processed data → saves `X_features_fe.parquet`.
4. **04_modeling.ipynb** → consumes engineered data → trains models.

---

### ✅ Benefits
- Reproducibility & version control  
- Integrity & traceability  
- Clear separation of pipeline stages  
- HR-ready professional structure  
